In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph google-generativeai "langchain[google-genai]" chromadb PyPdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 8.5 MB/s eta 0

**Components**

Chat model Setup: Google Gemini 2.5 Flash

In [ ]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

Enter API key for Google Gemini: ··········


Embedding Model setup: Gemini Embedding 001

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

**PART 2: Indexing**

In [ ]:
question = "Who's the ceo of Perplexity AI?"
document = "The CEO of Perplexity AI is Anand Srinivas."

Tokenization - Count the number of tokens

In [ ]:
import tiktoken

def num_of_string_tokens(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a String"""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

print(num_of_string_tokens(question, "cl100k_base"))


11


Text Embedding model

In [ ]:
from google import genai

clients = genai.Client()
query_result = clients.models.embed_content(model = "gemini-embedding-001", contents = question)
document_result = clients.models.embed_content(model = "gemini-embedding-001", contents = document)
print(document_result.embeddings[0])

values=[-0.013203796, -0.0027234135, 0.0034440185, -0.071691304, -0.021562928, -0.00042843062, -0.0076043257, 0.01339457, 0.02640388, -0.009397723, -0.0016088594, -0.03244042, 0.03373961, 0.02719173, 0.1026684, 0.0029305338, -0.010417919, 0.006685036, 0.022597773, 0.0013730156, 0.00068402145, 0.034946475, -0.025676921, -0.017784486, -0.021286547, -0.020180961, 0.021258995, -0.010923858, 0.03447221, 0.00921488, 0.019150535, -0.009102546, 0.000639536, 0.03833308, 0.04426743, -0.00010623932, 0.008631469, -0.014445077, 0.01605559, 0.013048882, -0.0035007996, -0.0065582073, -0.025243007, -0.013044004, 0.012513963, -0.024700262, -0.0043144766, 0.0026177133, -0.003910884, 0.024053723, -0.013788373, 0.008677988, -0.015915493, -0.1647992, -0.011156531, -0.001496209, 0.020851154, -0.026306616, 0.009348568, -0.027222097, -0.011415133, 0.009327651, -0.027335027, 0.011520506, -0.0048398497, -0.0117206555, -0.00048270705, 0.025434079, -0.001951213, 0.019277735, 0.008552485, 0.0024274434, 0.006737529

Cosine similarity b/w question and document

In [ ]:
import numpy as np

query_embedding = query_result.embeddings[0].values
doc_embedding = document_result.embeddings[0].values

cosine_sim = np.dot(query_embedding, doc_embedding) / (np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding))
print(cosine_sim)

0.8601693968224984


Document Loaders

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("unit_1_of_income_tax.pdf")

Income_docs = loader.load()

print(Income_docs[0].metadata)

{'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2017-09-21T10:30:50+05:30', 'author': 'Altaf', 'moddate': '2017-09-21T10:30:50+05:30', 'source': 'unit_1_of_income_tax.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}


Splitter

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) # Chunk overlap is used for continuation purposes b/w paragraphs. Preserves their relationship

# Make splits
all_splits = text_splitter.split_documents(Income_docs)
#print(all_splits)

In [ ]:
from langchain.embeddings.base import Embeddings
from google import genai
from langchain_community.vectorstores import Chroma

class GeminiEmbedding(Embeddings):
   def __init__(self, client):
       self.client = client

   def embed_documents(self, texts):
       return [self.client.models.embed_content(model="gemini-embedding-001", contents=text).embeddings[0].values for text in texts]

   def embed_query(self, text):
       return self.client.models.embed_content(model="gemini-embedding-001", contents=text).embeddings[0].values

# Then use:
clients = genai.Client()
gemini_embedding = GeminiEmbedding(clients)
vectorstore = Chroma.from_documents(documents=all_splits, embedding=gemini_embedding)
retriever = vectorstore.as_retriever()

**PART 3: RETRIEVAL**

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 1})

In [ ]:
docs = retriever.get_relevant_documents("When was tax first introduced in India?")

/tmp/ipython-input-685340943.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents("When was tax first introduced in India?")


In [ ]:
len(docs)

1

**PART 4: GENERATION**

In [ ]:
from langchain.prompts import ChatPromptTemplate

# Prompt
template = """
Answer the question based only on the context provided:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the context provided:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-1.5-flash-latest",
    temperature=0.6
)

In [ ]:
chain = prompt | llm

In [ ]:
chain.invoke({"context" : docs, "question" : "When was tax introduced to India?"})

AIMessage(content='Tax was first introduced to India in 1860.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--5fa601bc-4878-484e-9639-38376e4cdff0-0', usage_metadata={'input_tokens': 303, 'output_tokens': 14, 'total_tokens': 317, 'input_token_details': {'cache_read': 0}})